# DataCo Supply Chain Analysis
### Python + SQL Project (Google Colab)

This notebook analyzes the **DataCo Smart Supply Chain** sample dataset using **pandas** for data cleaning and **SQLite (SQL)** for querying business insights, with **matplotlib/seaborn** for visualization.

**What this notebook covers:**
1. Load & explore the data
2. Clean the data
3. Load data into a SQL (SQLite) database
4. Answer business questions with SQL queries
5. Visualize the results
6. Summary of insights

---
**How to use in Google Colab:**
1. Upload `dataco_sample.csv` when prompted in the cell below.
2. Run all cells: `Runtime > Run all`.


## 1. Setup & Upload Data

In [ ]:
# pandas, matplotlib, seaborn come pre-installed on Colab
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 60)


In [ ]:
# Upload the CSV file (run this cell, then choose dataco_sample.csv from your computer)
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab (e.g. local Jupyter) -- edit this path manually
    csv_path = "dataco_sample.csv"

print("Using file:", csv_path)


In [ ]:
# The dataset uses Latin-1 encoding (contains accented characters e.g. Mexico, Honduras)
df = pd.read_csv(csv_path, encoding="latin1")
print(df.shape)
df.head()


## 2. Explore the Data

Quick look at structure, data types, and missing values before cleaning.

In [ ]:
df.info()


In [ ]:
# Missing values per column (top 15)
df.isna().sum().sort_values(ascending=False).head(15)


In [ ]:
df.describe(include='number').T


## 3. Clean the Data

- Parse date columns
- Drop columns that carry no analytical value or are (fake/masked) PII in this sample (`Customer Password`, `Customer Email`, `Product Description`, `Product Image`)
- Standardize column names (lowercase, no spaces) so they're easy to reference in SQL


In [ ]:
df_clean = df.copy()

# Parse dates
df_clean["order_date"] = pd.to_datetime(df_clean["order date (DateOrders)"])
df_clean["shipping_date"] = pd.to_datetime(df_clean["shipping date (DateOrders)"])

# Drop columns not useful for this analysis
drop_cols = [
    "order date (DateOrders)", "shipping date (DateOrders)",
    "Customer Password", "Customer Email", "Product Description",
    "Product Image", "Customer Fname", "Customer Lname", "Customer Street"
]
df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

# Standardize column names: lowercase, underscores, strip special chars
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(df_clean.shape)
df_clean.head()


In [ ]:
# Check for duplicate rows and nulls in key business fields
print("Duplicate rows:", df_clean.duplicated().sum())
print()
print("Nulls in key columns:")
print(df_clean[["sales", "order_profit_per_order", "category_name", "customer_segment"]].isna().sum())


## 4. Normalize the Data & Load into SQL (SQLite)

The raw file is one flat table, so every query so far would just be filtering/aggregating a single sheet — no joins required. To actually practice **SQL joins**, we split two small dimension tables off the flat file before loading:

- **`customers`** — one row per `customer_id`, holding the descriptive customer attributes (segment, city, country).
- **`products`** — one row per `product_name`, holding the product's category, with a new surrogate key `product_id`.

The remaining columns (order-level facts: sales, profit, dates, shipping, etc.) stay in the **`orders`** table, referencing `customers` and `products` by foreign key (`customer_id`, `product_id`). This is the same star-schema pattern you'd see in a real data warehouse, and it means the business questions below have to `JOIN` tables back together to answer them — closer to real-world SQL work than querying one wide table.


In [ ]:
# ---- Split off a `customers` dimension table ----
customers_dim = (
    df_clean[["customer_id", "customer_segment", "customer_city", "customer_country"]]
    .drop_duplicates(subset="customer_id")
    .sort_values("customer_id")
    .reset_index(drop=True)
)

# ---- Split off a `products` dimension table ----
# The sample file has no natural product id, so we mint a surrogate key.
products_dim = (
    df_clean[["product_name", "category_name"]]
    .drop_duplicates(subset="product_name")
    .sort_values("product_name")
    .reset_index(drop=True)
)
products_dim.insert(0, "product_id", range(1, len(products_dim) + 1))

# ---- Build the `orders` fact table ----
# Drop the columns that now live in the dimension tables, keep FK ids to join back.
orders_fact = df_clean.merge(
    products_dim[["product_id", "product_name"]], on="product_name", how="left"
)
orders_fact = orders_fact.drop(
    columns=["customer_segment", "customer_city", "customer_country",
             "product_name", "category_name"]
)

print("customers:", customers_dim.shape)
print("products: ", products_dim.shape)
print("orders:   ", orders_fact.shape)
customers_dim.head(3)


In [ ]:
conn = sqlite3.connect(":memory:")
orders_fact.to_sql("orders", conn, index=False, if_exists="replace")
customers_dim.to_sql("customers", conn, index=False, if_exists="replace")
products_dim.to_sql("products", conn, index=False, if_exists="replace")

# Quick sanity check
pd.read_sql_query("SELECT COUNT(*) AS total_orders FROM orders;", conn)


In [ ]:
# See the table schema SQLite generated for each of the three tables
for table in ["orders", "customers", "products"]:
    print(f"--- {table} ---")
    schema = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print(schema[["name", "type"]].to_string(index=False))
    print()


## 5. Business Questions Answered with SQL

Each section below runs a SQL query against the `orders` table and visualizes the result.


### 5.1 Overall KPIs

In [ ]:
query = '''
SELECT
    COUNT(DISTINCT order_id)      AS total_orders,
    COUNT(DISTINCT customer_id)   AS total_customers,
    ROUND(SUM(sales), 2)          AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(order_profit_per_order), 2) AS avg_profit_per_order
FROM orders;
'''
kpis = pd.read_sql_query(query, conn)
kpis


### 5.2 Sales & Profit by Product Category

In [ ]:
query = '''
SELECT
    p.category_name,
    COUNT(*)                              AS num_orders,
    ROUND(SUM(o.sales), 2)                AS total_sales,
    ROUND(SUM(o.order_profit_per_order), 2) AS total_profit
FROM orders AS o
JOIN products AS p ON o.product_id = p.product_id
GROUP BY p.category_name
ORDER BY total_sales DESC
LIMIT 10;
'''
category_sales = pd.read_sql_query(query, conn)
category_sales


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=category_sales, y="category_name", x="total_sales", palette="viridis")
plt.title("Top 10 Product Categories by Total Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


### 5.3 Top 10 Best-Selling Products

In [ ]:
query = '''
SELECT
    p.product_name,
    SUM(o.order_item_quantity)              AS units_sold,
    ROUND(SUM(o.sales), 2)                  AS total_sales,
    ROUND(SUM(o.order_profit_per_order), 2) AS total_profit
FROM orders AS o
JOIN products AS p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_sales DESC
LIMIT 10;
'''
top_products = pd.read_sql_query(query, conn)
top_products


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top_products, y="product_name", x="total_sales", palette="mako")
plt.title("Top 10 Best-Selling Products by Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


### 5.4 Sales by Market / Region

In [ ]:
query = '''
SELECT
    market,
    order_region,
    ROUND(SUM(sales), 2) AS total_sales,
    COUNT(*)             AS num_orders
FROM orders
GROUP BY market, order_region
ORDER BY total_sales DESC
LIMIT 10;
'''
region_sales = pd.read_sql_query(query, conn)
region_sales


In [ ]:
market_totals = pd.read_sql_query('''
    SELECT market, ROUND(SUM(sales),2) AS total_sales
    FROM orders GROUP BY market ORDER BY total_sales DESC;
''', conn)

plt.figure(figsize=(7, 7))
plt.pie(market_totals["total_sales"], labels=market_totals["market"], autopct="%1.1f%%",
        startangle=90, colors=sns.color_palette("Set2"))
plt.title("Share of Total Sales by Market")
plt.tight_layout()
plt.show()


### 5.5 Late Delivery Risk by Shipping Mode

In [ ]:
query = '''
SELECT
    shipping_mode,
    COUNT(*)                                            AS total_orders,
    SUM(late_delivery_risk)                              AS late_orders,
    ROUND(100.0 * SUM(late_delivery_risk) / COUNT(*), 1) AS late_pct
FROM orders
GROUP BY shipping_mode
ORDER BY late_pct DESC;
'''
late_by_mode = pd.read_sql_query(query, conn)
late_by_mode


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=late_by_mode, x="shipping_mode", y="late_pct", palette="rocket")
plt.title("Late Delivery Rate (%) by Shipping Mode")
plt.ylabel("Late Delivery Rate (%)")
plt.xlabel("Shipping Mode")
plt.tight_layout()
plt.show()


### 5.6 Monthly Sales Trend

In [ ]:
query = '''
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    ROUND(SUM(sales), 2)          AS total_sales,
    ROUND(SUM(order_profit_per_order), 2) AS total_profit
FROM orders
GROUP BY order_month
ORDER BY order_month;
'''
monthly = pd.read_sql_query(query, conn)
monthly


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly["order_month"], monthly["total_sales"], marker="o", label="Sales")
plt.plot(monthly["order_month"], monthly["total_profit"], marker="o", label="Profit")
plt.xticks(rotation=45)
plt.title("Monthly Sales & Profit Trend")
plt.xlabel("Month")
plt.ylabel("Amount ($)")
plt.legend()
plt.tight_layout()
plt.show()


### 5.7 Top 10 Customers by Revenue

In [ ]:
query = '''
SELECT
    c.customer_id,
    c.customer_segment,
    c.customer_city,
    c.customer_country,
    ROUND(SUM(o.sales), 2) AS total_spent,
    COUNT(DISTINCT o.order_id) AS num_orders
FROM orders AS o
JOIN customers AS c ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_segment, c.customer_city, c.customer_country
ORDER BY total_spent DESC
LIMIT 10;
'''
top_customers = pd.read_sql_query(query, conn)
top_customers


### 5.8 Profitability by Customer Segment

In [ ]:
query = '''
SELECT
    c.customer_segment,
    COUNT(DISTINCT o.order_id)              AS num_orders,
    ROUND(SUM(o.sales), 2)                  AS total_sales,
    ROUND(SUM(o.order_profit_per_order), 2) AS total_profit,
    ROUND(AVG(o.order_item_profit_ratio), 3) AS avg_profit_ratio
FROM orders AS o
JOIN customers AS c ON o.customer_id = c.customer_id
GROUP BY c.customer_segment
ORDER BY total_sales DESC;
'''
segment_profit = pd.read_sql_query(query, conn)
segment_profit


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=segment_profit, x="customer_segment", y="total_sales", ax=axes[0], palette="crest")
axes[0].set_title("Total Sales by Customer Segment")
sns.barplot(data=segment_profit, x="customer_segment", y="total_profit", ax=axes[1], palette="flare")
axes[1].set_title("Total Profit by Customer Segment")
plt.tight_layout()
plt.show()


### 5.9 Order Status Breakdown

In [ ]:
query = '''
SELECT order_status, COUNT(*) AS num_orders
FROM orders
GROUP BY order_status
ORDER BY num_orders DESC;
'''
order_status = pd.read_sql_query(query, conn)

plt.figure(figsize=(9, 5))
sns.barplot(data=order_status, y="order_status", x="num_orders", palette="cubehelix")
plt.title("Order Count by Order Status")
plt.tight_layout()
plt.show()


## 6. Summary of Insights

Run the notebook end-to-end, then fill in your own takeaways here based on the query results above, for example:

- Which product category drives the most sales/profit?
- Which shipping mode has the highest late-delivery risk?
- Are sales trending up or down month-over-month?
- Which customer segment is most profitable?

## 7. Next Steps (Optional Extensions)

- Persist the SQLite DB to a file (`sqlite3.connect("dataco.db")`) instead of `:memory:` so it survives between sessions.
- Add a **discount impact analysis**: does `order_item_discount_rate` correlate with `order_item_profit_ratio`?
- Build a simple **RFM (Recency, Frequency, Monetary) customer segmentation** using SQL window functions.
- Try connecting this notebook to **Google Sheets/BigQuery** for a live dashboard.
- Extend the star schema further (e.g. split off a `shipping` or `location` dimension) and practice `LEFT JOIN`s to check for orders with no matching customer/product.
